In [1]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

## Functions

In [ ]:
class Rosenbrock:
    def __init__(self):
        self.name = "Rosenbrock"
        self.minima= (1, 1) # Global minimum of Rosenbrock function
        self.limits = [(-2, 2), (-1, 3)] # Limits for Rosenbrock function

    def evaluate(self, x, y):
        return (20 + (x**2 - 10*np.cos(2*np.pi*x)) + (y**2 - 10*np.cos(2*np.pi*y))) # Rosenbrock function 

    #def evaluate_with_error(self, x, y, err: float=0.0):
    #    # l'error es rep com a float. Es a dir, un 20% d'error correspont a err=0.2
    #    error_random = XXX(1+err, 1-err)
    #    return error_random*((1 - x)**2 + 100*(y - (x**2))**2)

class Rastrigin: 
    def __init__(self):
        self.name = "Rastrigin"
        self.minima = [(0, 0)] # Global minimum of Rastrigin function
        self.limits = (-5.12, 5.12) # Limits for Rastrigin function

    def evaluate(self, x, y):
        return (20 + (x**2 - 10*np.cos(2*np.pi*x)) + (y**2 - 10*np.cos(2*np.pi*y))) # Rastrigin function
    
class Himmelblau:
    def __init__(self):
        self.name   = "Himmelblau"
        self.minima = [(3, 2), (-2.805, 3.131), (-3.779, -3.283), (3.584, -1.848)] # Global minima of Himmelblau function
        self.limits = [(-6, 6), (-6, 6)] # Limits for Himmelblau function

    def evaluate(self, x, y):
        return (x**2 + y - 11)**2 + (x + y**2 - 7)**2 # Himmelblau function
    
class Beale:
    def __init__(self):
        self.name   = "Beale"
        self.minima = [(3, 0.5)] # Global minimum of Beale function
        self.limits = [(-4, 4), (-4, 4)] # Limits for Beale function

    def evaluate(self, x, y):
        return (1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2 # Beale function

In [ ]:
def lectura(filepath: str):
    df = pd.read_csv(filepath, sep=None, engine='python') 
    df.columns = df.columns.str.strip() #Clean column names by stripping whitespace
    print(f"Detected columns: {list(df.columns)}")

    val_x = df.iloc[:, 1] # Extract values from the second column (X) using .iloc, which is where the X values are located
    val_y = df.iloc[:, 2] # Extract values from the third column (Y) using .iloc, which is where the Y values are located

    return val_x, val_y, df

In [ ]:
def plot_round(val_x, val_y, funcio: function, round: str): #, save_figure: bool = False): 

    ###-PLOT-###
    # Create the background map of the function 
    x_range = np.linspace(funcio.limits[0][0], funcio.limits[0][1], 400)
    y_range = np.linspace(funcio.limits[1][0], funcio.limits[1][1], 400)
    X_mesh, Y_mesh = np.meshgrid(x_range, y_range)
    Z_mesh = funcio.evaluate(X_mesh, Y_mesh)

    # Create the contour plot
    plt.figure(figsize=(10, 7))
    plt.contourf(X_mesh, Y_mesh, Z_mesh, levels=np.logspace(-2, 5, 40), norm=LogNorm(vmin=0.01, vmax=10**5), cmap='viridis', alpha=0.8)
    plt.colorbar(label=f'{funcio.name} value')

    # Draw the explored points
    plt.scatter(val_x, val_y, color='red', edgecolors='white', s=80, label='Explored points')

    # Draw the white contour lines of the points
    #plt.contour(X_mesh, Y_mesh, Z_mesh, levels=15, colors='white', linewidths=0.5, alpha=0.8)

    # Add labels to the points
    #for i, row in df.iterrows():
    #    plt.text(row.iloc[1]+0.1, row.iloc[2]+0.1, str(int(row.iloc[0])), color='white', fontsize=9)

    plt.title(f"Round {round}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()

    return plt


NameError: name 'function' is not defined

## Main

In [ ]:
def avaluacio(funcio, folder_path, save_plot: bool=False):

    csvs = [f for f in os.listdir(folder_path) if f.endswith('.csv') and f.startswith(f'{funcio.name}') and "RESULT" not in f and "plot" not in f]
    csvs = sorted(csvs, key=lambda f: int(f.split('_')[-1].replace('.csv', '').replace('R', '')))
    for file in csvs:
        full_path = os.path.join(folder_path, file)
        output_file = file.replace('.csv', '_RESULT.csv')
        plot_file   = file.replace('.csv', '_plot.jpg')
        round       = int(file.split('_')[-1].replace('.csv', '').replace('R', '')) 

        print("-" * 30)
        print(f"Processing file: {file}")
        print(f"Saving output to {output_file}")
        print(f"Saving plot to {plot_file}")
        print(f"Round: {round}")
        print("-" * 30)

        val_x, val_y, df = lectura(full_path)

    
        df['real'] = funcio.evaluate(val_x, val_y) # Calculate the function value for each (X, Y) pair and store it in a new column called 'real'


        df.to_csv(folder_path+output_file, index=False)

        print(df.head()) 
        plt = plot_round(val_x, val_y, funcio, str(round))

        if save_plot:
            plt.savefig(folder_path+plot_file, dpi=300, bbox_inches='tight')
            print(f"Plot saved correctly as: {plot_file}")

## MAE + Distance values

In [ ]:
def metrics(funcio, folder_path):

    csvs_results = [f for f in os.listdir(folder_path) if f.endswith('RESULT.csv') and f.startswith(f'{funcio.name}')]
    csvs_results = sorted(csvs_results, key=lambda f: int(f.split('_')[-2].replace('.csv', '').replace('R', '')))

    resum_csv = []
    for file in csvs_results:
        file_path = os.path.join(folder_path, file)

        val_x, val_y, df = lectura(file_path) ## llegim les dades
        #val_real = funcio.evaluate(val_x, val_y) ## calcul del valor REAl de la funcio per a cada punt

        # Calculate the distance of each point to the global minimum of the function and store these distances in new columns in the DataFrame
        for i, (mx, my) in enumerate(funcio.minima, 1):
            df[f'M{i}'] = np.sqrt((val_x - mx)**2 + (val_y - my)**2)

        # Calculate the MAE and the total mean 
        if 'R1' in df.columns:
            df['Punctual_error'] = np.abs(df['R1'] - df['real'])
            final_mae = df['Punctual_error'].mean()
        else:
            final_mae = 0 # Per si és la R0 i no hi ha predicció encara

        round_name = file.split('_')[1] 

        fila_resum = {
            'ROUNDS': round_name,
            'MAE': final_mae,
            'Distance': df['M1'].min() 
        }

        resum_csv.append(fila_resum)

        print(f"Funcio: {funcio.name}. Ronda: {round_name}. MAE: {final_mae:.4f}")

    # Final CSV with all results
    df_final = pd.DataFrame(resum_csv)

    # Opcional: Ordenar les rondes numèricament perquè el gràfic no surti desordenat
    #df_final['n'] = df_final['ROUNDS'].str.extract('(\d+)').astype(int)
    #df_final = df_final.sort_values('n').drop(columns=['n'])

    # Save the file that the plot code will read
    save_path = os.path.join(folder_path, f"Summary_{funcio.name}.csv")
    df_final.to_csv(save_path, index=False)

    print("\n" + "="*40)
    print(f"File saved correctly to:\n{save_path}")
    print("="*40)
    print(df_final.to_string(index=False))

## Plots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

BASE_DIR = os.getcwd()
folder_path = BASE_DIR
input_consolidat = os.path.join(folder_path, "Summary_Table_Rosenbrock.csv")


def plots(funcio, folder_path):

    if os.path.exists(input_consolidat):
        df = pd.read_csv(input_consolidat)
        df['round_number'] = df['ROUNDS'].str.replace('R', '').astype(int)
        df = df.sort_values('round_number')

        plt.rcParams['font.family'] = 'sans-serif'
        color_sampling = '#219ebc' 
        color_fitting = '#6a1b9a'

        # =========================================================
        # --- PLOT 1: SAMPLING ---
        # =========================================================
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # MODIFICACIÓ AQUÍ: Només una columna i una etiqueta
        ax1.plot(df['round_number'], df['Distance'], 
                 marker='o', markersize=8, markerfacecolor='white', markeredgewidth=2,
                 linestyle='-', linewidth=3, color=color_sampling, 
                 label='Global Minimum (3, 0.5)', alpha=0.9)

        ax1.set_title('Space exploration (Sampling) - Rosenbrock', loc='left', fontsize=16, fontweight='bold', pad=20)
        ax1.set_xticks(df['round_number'])
        ax1.set_xlabel('Round', fontsize=12)
        ax1.set_ylabel('Distance to Minimum', fontsize=12)

        ax1.legend(frameon=False)
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        ax1.grid(axis='y', linestyle='--', alpha=0.3) 

        plt.tight_layout()
        plt.show()

        # =========================================================
        # --- PLOT 2: FITTING (MAE) ---
        # =========================================================
        fig, ax2 = plt.subplots(figsize=(10, 5))
        ax2.fill_between(df['round_number'], df['MAE'], color=color_fitting, alpha=0.1)
        ax2.plot(df['round_number'], df['MAE'], marker='s', markersize=7, color=color_fitting, linewidth=3)

        ax2.set_title('Model convergence (Fitting) - Rosenbrock', loc='left', fontsize=16, fontweight='bold', pad=20)
        ax2.set_xticks(df['round_number'])
        ax2.set_xlabel('Round', fontsize=12)
        ax2.set_ylabel('Mean Absolute Error (MAE)', fontsize=12)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)

        final_mae = df['MAE'].iloc[-1]
        ax2.annotate(f'Final: {final_mae:.4f}', 
                     xy=(df['round_number'].iloc[-1], final_mae),
                     xytext=(-60, 20), textcoords='offset points',
                     arrowprops=dict(arrowstyle='->', color='#555555'))

        plt.tight_layout()
        plt.show()
    else:
        print(f"Error: No s'ha trobat {input_consolidat}.")

In [7]:
funcions = [Himmelblau(), Rosenbrock(), Beale(), Rastrigin()]

for funcio in funcions:
    BASE_DIR = os.getcwd()
    folder_path = BASE_DIR+f"/{funcio.name}/"

    avaluacio(funcio, folder_path)
    metriques(funcio, folder_path)
    plots(funcio, folder_path)


FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'C:\\Users\\Usuario\\AppData\\Local\\Programs\\Microsoft VS Code/Himmelblau/'